## Импорт библиотек

In [45]:
import warnings
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

from datetime import datetime

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense


In [46]:
warnings.filterwarnings('ignore')

In [47]:
seed = 42
np.random.seed(seed)
random.seed(seed)

## Загрузка данных

In [48]:
df = pd.read_csv('df_for_learning.csv', index_col=0)
df.head()

,Индекс ВМО,date,Средняя температура воздуха,Количество осадков,Горизонтальная дальность видимости,Общее количество облачности,Количество облачности нижнего яруса,Форма облаков верхнего яруса,Форма облаков среднего яруса,Форма облаков вертикального развития,...,Атмосферное давление на уровне станции_lag_395,year,day,day_of_year_sin,day_of_year_cos,season,month_sin,month_cos,Средняя температура воздуха_mean_group,temp_delta_vs_all_stations
0,34059,1982-01-31,-9.2,0.0,69.125,5.125,0.0,0.750,0.000,0.0,...,983.0500,1982,31,0.508671,0.860961,1,0.500000,0.866025,-8.820930,0.110631
1,34059,1982-02-01,-11.8,0.0,68.125,3.375,0.0,1.750,0.125,0.0,...,980.3750,1982,1,0.523416,0.852078,1,0.866025,0.500000,-8.623256,0.519601
2,34059,1982-02-02,-15.2,0.0,67.625,1.000,0.0,0.375,0.000,0.0,...,984.6000,1982,2,0.538005,0.842942,1,0.866025,0.500000,-9.165116,0.276744
3,34059,1982-02-03,-16.3,0.0,53.500,0.625,0.0,0.250,0.000,0.0,...,981.3750,1982,3,0.552435,0.833556,1,0.866025,0.500000,-9.504651,0.185382
4,34059,1982-02-04,-17.4,0.0,55.000,5.000,0.0,2.500,0.625,0.0,...,979.3875,1982,4,0.566702,0.823923,1,0.866025,0.500000,-8.655814,0.614618


In [49]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['Индекс ВМО', 'date']).reset_index(drop=True)

In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109053 entries, 0 to 109052
Data columns (total 57 columns):
 #   Column                                          Non-Null Count   Dtype         
---  ------                                          --------------   -----         
 0   Индекс ВМО                                      109053 non-null  int64         
 1   date                                            109053 non-null  datetime64[ns]
 2   Средняя температура воздуха                     109053 non-null  float64       
 3   Количество осадков                              109053 non-null  float64       
 4   Горизонтальная дальность видимости              109053 non-null  float64       
 5   Общее количество облачности                     109053 non-null  float64       
 6   Количество облачности нижнего яруса             109053 non-null  float64       
 7   Форма облаков верхнего яруса                    109053 non-null  float64       
 8   Форма облаков среднего яруса      

## Работа с лагами

### Выбор признаков

In [51]:
df.shape

(109053, 57)

In [52]:
original_cols = [
    col for col in df.columns
    if not any([
        '_lag_' in col,
        '_sin' in col,
        '_cos' in col,
        '_mean' in col,
        'delta' in col,
        col in ['year', 'day', 'season', 'Индекс ВМО', 'date']
    ])
]
# original_cols

### Добавление лагов

In [53]:
def add_lag_features(df, lag_cols, lag=1, group_col='Индекс ВМО', date_col='date'):
    """
    Добавляет лаги для указанных колонок
    df : DataFrame
    lag_cols : list - колонки, для которых делаем лаги
    lag : int - размер лага (в днях)
    """
    df = df.sort_values([group_col, date_col]).copy()

    for col in lag_cols:
        df[f'{col}_lag_{lag}'] = df.groupby(group_col)[col].shift(lag)

    return df

In [54]:
df_with_lags = add_lag_features(df, original_cols, lag=5)

In [55]:
df_with_lags.columns

Index(['Индекс ВМО', 'date', 'Средняя температура воздуха',
       'Количество осадков', 'Горизонтальная дальность видимости',
       'Общее количество облачности', 'Количество облачности нижнего яруса',
       'Форма облаков верхнего яруса', 'Форма облаков среднего яруса',
       'Форма облаков вертикального развития',
       'Слоистые и слоисто_кучевые облака',
       'Слоисто_дождразорванно_дождевые облака',
       'Высота нижней границы облачности', 'Погода между сроками',
       'Погода в срок наблюдения', 'Направление ветра',
       'Средняя скорость ветра', 'Максимальная скорость ветра',
       'Сумма осадков', 'Температура поверхности почвы',
       'Парциальное давление водяного пара', 'Относительная влажность воздуха',
       'Дефицит насыщения водяного пара',
       'Атмосферное давление на уровне станции',
       'Атмосферное давление на уровне моря',
       'Характеристика барической тенденции', 'Величина барической тенденции',
       'Средняя температура воздуха_lag_335',

### Работа с пропусками

In [56]:
df_with_lags.isna().sum().sum()

np.int64(875)

In [57]:
df_with_lags.date.min(), df_with_lags.date.max()

(Timestamp('1982-01-31 00:00:00'), Timestamp('2024-09-25 00:00:00'))

In [58]:
# Для каждой группы находим первую дату без NaN
def first_valid_date_per_group(df, group_col='Индекс ВМО', date_col='date'):
    # Словарь: индекс ВМО - дата первой строки без NaN
    first_dates = {}

    for idx, group in df.groupby(group_col):
        # Проверяем только колонки с NaN
        valid_rows = group[~group.isna().any(axis=1)]
        if not valid_rows.empty:
            first_dates[idx] = valid_rows[date_col].iloc[0]
        else:
            first_dates[idx] = None  # если все строки пустые

    return first_dates


first_dates_dict = first_valid_date_per_group(df_with_lags)
first_dates_dict

{34059: Timestamp('1982-02-05 00:00:00'),
 34152: Timestamp('1982-02-05 00:00:00'),
 34163: Timestamp('1982-02-05 00:00:00'),
 34172: Timestamp('1982-02-05 00:00:00'),
 34186: Timestamp('1982-02-05 00:00:00'),
 34289: Timestamp('1982-02-05 00:00:00'),
 34391: Timestamp('1982-02-05 00:00:00')}

In [59]:
# Обрезаем каждую группу по дате
cleaned_groups = []
for idx, group in df_with_lags.groupby('Индекс ВМО'):
    start_date = first_dates_dict[idx]
    cleaned_group = group[group['date'] >= start_date]
    cleaned_groups.append(cleaned_group)

clean_lagged_df = pd.concat(cleaned_groups).reset_index(drop=True)

In [60]:
clean_lagged_df.date.min(), clean_lagged_df.date.max()

(Timestamp('1982-02-05 00:00:00'), Timestamp('2024-09-25 00:00:00'))

In [61]:
clean_lagged_df.isna().sum().sum()

np.int64(0)

In [62]:
df.shape, clean_lagged_df.shape

((109053, 57), (109018, 82))

## Удаление лишних признаков (Чтобы не было утечек)

Делать предсказания по предыдущим дням, а не по текущему восстанавливать

In [63]:
columns_for_drop = [col for col in original_cols if col != 'Средняя температура воздуха']

In [64]:
df_for_training = clean_lagged_df.drop(columns=columns_for_drop)

In [65]:
df_for_training.head()

,Индекс ВМО,date,Средняя температура воздуха,Средняя температура воздуха_lag_335,Средняя температура воздуха_lag_365,Средняя температура воздуха_lag_395,Температура поверхности почвы_lag_335,Температура поверхности почвы_lag_365,Температура поверхности почвы_lag_395,Парциальное давление водяного пара_lag_335,...,Максимальная скорость ветра_lag_5,Сумма осадков_lag_5,Температура поверхности почвы_lag_5,Парциальное давление водяного пара_lag_5,Относительная влажность воздуха_lag_5,Дефицит насыщения водяного пара_lag_5,Атмосферное давление на уровне станции_lag_5,Атмосферное давление на уровне моря_lag_5,Характеристика барической тенденции_lag_5,Величина барической тенденции_lag_5
0,34059,1982-02-05,-7.9,-5.7,-0.7,0.4,-9.6250,-1.3875,-0.3750,3.0125,...,10.000,0.0,-10.750,2.1125,75.375,0.7000,997.0250,1018.3125,5.125,0.7125
1,34059,1982-02-06,-9.8,-11.1,2.4,-1.8,-6.8875,1.0000,-2.0125,3.7750,...,7.125,0.0,-12.500,1.8875,77.375,0.5625,1001.1625,1022.6250,2.000,0.9750
2,34059,1982-02-07,-18.2,-1.9,1.9,-8.6,-2.2750,0.0000,-8.5000,4.1750,...,3.500,0.0,-16.750,1.3250,73.125,0.5250,1006.4125,1028.3000,3.500,0.4125
3,34059,1982-02-08,-20.0,-1.7,0.8,-13.3,-1.4000,0.0625,-14.0000,5.9500,...,4.750,0.0,-17.875,1.3750,79.125,0.3625,1005.8750,1027.8500,4.000,0.4875
4,34059,1982-02-09,-22.0,-0.8,-1.3,-13.8,-3.1625,-1.2625,-11.2500,3.9000,...,2.750,0.0,-16.125,1.5375,78.875,0.3750,1004.3125,1026.1750,6.375,0.6500


In [66]:
df_for_training.shape

(109018, 58)

In [67]:
df_for_training.isna().sum().sum()

np.int64(0)

## Разделение данных на тренировочную/валидационную

In [68]:
df_for_training.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109018 entries, 0 to 109017
Data columns (total 58 columns):
 #   Column                                          Non-Null Count   Dtype         
---  ------                                          --------------   -----         
 0   Индекс ВМО                                      109018 non-null  int64         
 1   date                                            109018 non-null  datetime64[ns]
 2   Средняя температура воздуха                     109018 non-null  float64       
 3   Средняя температура воздуха_lag_335             109018 non-null  float64       
 4   Средняя температура воздуха_lag_365             109018 non-null  float64       
 5   Средняя температура воздуха_lag_395             109018 non-null  float64       
 6   Температура поверхности почвы_lag_335           109018 non-null  float64       
 7   Температура поверхности почвы_lag_365           109018 non-null  float64       
 8   Температура поверхности почвы_lag_

In [69]:
df_for_training.date.min(), df_for_training.date.max()

(Timestamp('1982-02-05 00:00:00'), Timestamp('2024-09-25 00:00:00'))

In [70]:
val_start = df_for_training['date'].max() - pd.DateOffset(months=1)
val_start

Timestamp('2024-08-25 00:00:00')

In [71]:
# Разделяем
train_df = df_for_training[df_for_training['date'] < val_start].copy()
val_df = df_for_training[df_for_training['date'] >= val_start].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

Train shape: (108794, 58)
Validation shape: (224, 58)


In [72]:
train_df = train_df.drop(columns=['date'])
val_df_drop_date = val_df.drop(columns=['date'])

In [73]:
train_df.columns

Index(['Индекс ВМО', 'Средняя температура воздуха',
       'Средняя температура воздуха_lag_335',
       'Средняя температура воздуха_lag_365',
       'Средняя температура воздуха_lag_395',
       'Температура поверхности почвы_lag_335',
       'Температура поверхности почвы_lag_365',
       'Температура поверхности почвы_lag_395',
       'Парциальное давление водяного пара_lag_335',
       'Парциальное давление водяного пара_lag_365',
       'Парциальное давление водяного пара_lag_395',
       'Относительная влажность воздуха_lag_335',
       'Относительная влажность воздуха_lag_365',
       'Относительная влажность воздуха_lag_395',
       'Дефицит насыщения водяного пара_lag_335',
       'Дефицит насыщения водяного пара_lag_365',
       'Дефицит насыщения водяного пара_lag_395',
       'Атмосферное давление на уровне моря_lag_335',
       'Атмосферное давление на уровне моря_lag_365',
       'Атмосферное давление на уровне моря_lag_395',
       'Атмосферное давление на уровне станци

In [74]:
y_train = train_df['Средняя температура воздуха']
y_val = val_df_drop_date['Средняя температура воздуха']

# Признаки (оставляем Индекс ВМО)
X_train = train_df.drop(columns=['Средняя температура воздуха'])
X_val = val_df_drop_date.drop(columns=['Средняя температура воздуха'])

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)

Train shape: (108794, 56) (108794,)
Validation shape: (224, 56) (224,)


In [75]:
print(X_train.isna().sum().sum())
print(y_train.isna().sum().sum())
print('Валидационное:')
print(X_val.isna().sum().sum())
print(y_val.isna().sum().sum())

0
0
Валидационное:
0
0


In [76]:
X_train.columns

Index(['Индекс ВМО', 'Средняя температура воздуха_lag_335',
       'Средняя температура воздуха_lag_365',
       'Средняя температура воздуха_lag_395',
       'Температура поверхности почвы_lag_335',
       'Температура поверхности почвы_lag_365',
       'Температура поверхности почвы_lag_395',
       'Парциальное давление водяного пара_lag_335',
       'Парциальное давление водяного пара_lag_365',
       'Парциальное давление водяного пара_lag_395',
       'Относительная влажность воздуха_lag_335',
       'Относительная влажность воздуха_lag_365',
       'Относительная влажность воздуха_lag_395',
       'Дефицит насыщения водяного пара_lag_335',
       'Дефицит насыщения водяного пара_lag_365',
       'Дефицит насыщения водяного пара_lag_395',
       'Атмосферное давление на уровне моря_lag_335',
       'Атмосферное давление на уровне моря_lag_365',
       'Атмосферное давление на уровне моря_lag_395',
       'Атмосферное давление на уровне станции_lag_335',
       'Атмосферное давлен

## Функции оценки точности

### Стандартные методы


In [77]:
def simple_metric(y_val, y_pred):
    mae = mean_absolute_error(y_val, y_pred)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_val - y_pred) / y_val)) * 100
    r2 = r2_score(y_val, y_pred)

    print("Простые метрики:")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"R²: {r2:.4f}")
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2
    }

### Ошибки по дням

In [78]:
def daily_metrics(df_val, y_pred, date_col='date',
                  y_true_col='Средняя температура воздуха', y_pred_col='prediction'):
    df_val['prediction'] = y_pred
    results = []

    for date, group in df_val.groupby(date_col):
        y_true = group[y_true_col].values
        y_pred = group[y_pred_col].values

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)

        results.append({
            'date': date,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2
        })

    return pd.DataFrame(results).sort_values('date')

### Столбчатые диаграммы ошибок по индексам

In [79]:
def metrics_per_index(df_val, y_pred, index_col='Индекс ВМО',
                      y_true_col='Средняя температура воздуха', y_pred_col='prediction'):
    """
    df_val: DataFrame с колонками y_true и индекс
    y_pred: массив предсказаний
    """
    df_val = df_val.copy()
    df_val[y_pred_col] = y_pred

    metrics_list = []

    for idx, group in df_val.groupby(index_col):
        y_true = group[y_true_col].values
        y_p = group[y_pred_col].values

        mae = mean_absolute_error(y_true, y_p)
        rmse = np.sqrt(mean_squared_error(y_true, y_p))
        r2 = r2_score(y_true, y_p)

        metrics_list.append({
            index_col: idx,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2
        })

    metrics_df = pd.DataFrame(metrics_list).sort_values(index_col)

    # Строим графики
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

    axes[0].bar(metrics_df[index_col].astype(str), metrics_df['MAE'], color='skyblue')
    axes[0].set_title('MAE по индексам')
    axes[0].set_ylabel('MAE (°C)')

    axes[1].bar(metrics_df[index_col].astype(str), metrics_df['RMSE'], color='salmon')
    axes[1].set_title('RMSE по индексам')
    axes[1].set_ylabel('RMSE (°C)')

    axes[2].bar(metrics_df[index_col].astype(str), metrics_df['R2'], color='lightgreen')
    axes[2].set_title('R² по индексам')
    axes[2].set_ylabel('R²')

    for ax in axes:
        ax.set_xlabel('Индекс ВМО')
        ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    return metrics_df

### Линейный график MAE

In [80]:
def cumulative_mae(df_val, y_pred, date_col='date',
                   y_true_col='Средняя температура воздуха', y_pred_col='prediction'):
    """
    df_val: DataFrame с колонками y_true и date
    y_pred: массив предсказаний
    """
    df_val = df_val.copy()
    df_val[y_pred_col] = y_pred

    # Сортируем по дате
    df_val = df_val.sort_values(date_col)

    # Копим MAE по дням
    cum_errors = []
    all_true = []
    all_pred = []

    for date, group in df_val.groupby(date_col):
        all_true.extend(group[y_true_col].values)
        all_pred.extend(group[y_pred_col].values)

        mae = mean_absolute_error(all_true, all_pred)
        cum_errors.append({'date': date, 'Cumulative_MAE': mae})

    cum_df = pd.DataFrame(cum_errors)

    # График
    plt.figure(figsize=(12, 5))
    plt.plot(cum_df['date'], cum_df['Cumulative_MAE'], marker='o', linestyle='-')
    plt.xlabel('Дата')
    plt.ylabel('MAE (°C)')
    plt.title('MAE по дням')
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()

    return cum_df

### Накопление ошибки Mae

In [81]:
def cumulative_error(df_val, y_pred, date_col='date',
                     y_true_col='Средняя температура воздуха', y_pred_col='prediction'):
    df_val = df_val.copy()
    df_val[y_pred_col] = y_pred

    df_val = df_val.sort_values(date_col)

    cum_errors = []
    total_error = 0

    for date, group in df_val.groupby(date_col):
        daily_error = (group[y_true_col] - group[y_pred_col]).abs().sum()
        total_error += daily_error

        cum_errors.append({'date': date, 'Cumulative_Error': total_error})

    cum_df = pd.DataFrame(cum_errors)

    # График
    plt.figure(figsize=(12, 5))
    plt.plot(cum_df['date'], cum_df['Cumulative_Error'], marker='o')
    plt.xlabel('Дата')
    plt.ylabel('Накопленная ошибка')
    plt.title('Накопление абсолютной ошибки по дням')
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()

    return cum_df

### Сравнения истинной средней и предсказанной средней

In [82]:
def plot_true_vs_pred(df_val, y_pred, date_col='date', y_true_col='Средняя температура воздуха'):
    df_plot = df_val.copy()
    df_plot['prediction'] = y_pred

    # Усредняем по индексам
    daily_true = df_plot.groupby(date_col)[y_true_col].mean()
    daily_pred = df_plot.groupby(date_col)['prediction'].mean()

    plt.figure(figsize=(12, 6))
    plt.plot(daily_true.index, daily_true.values, label='Истинная температура', marker='o', linestyle='-', color='blue')
    plt.plot(daily_pred.index, daily_pred.values, label='Предсказанная температура', marker='x', linestyle='--',
             color='orange')
    plt.xlabel('Дата')
    plt.ylabel('Температура (°C)')
    plt.title('Сравнение истинной и предсказанной температуры')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### Сравнение истинной и предсказанной по индексам

In [83]:
def plot_true_vs_pred_by_index(df_val, y_pred, date_col='date', index_col='Индекс ВМО',
                               y_true_col='Средняя температура воздуха'):
    df_plot = df_val.copy()
    df_plot['prediction'] = y_pred

    indices = df_plot[index_col].unique()
    n_indices = len(indices)

    # Определяем размер подграфиков (по 2 в ряду)
    n_cols = 2
    n_rows = (n_indices + 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3), sharex=True, sharey=True)
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        ax = axes[i]
        df_idx = df_plot[df_plot[index_col] == idx]
        ax.plot(df_idx[date_col], df_idx[y_true_col], label='Истинная', marker='o', linestyle='-', color='blue')
        ax.plot(df_idx[date_col], df_idx['prediction'], label='Предсказанная', marker='x', linestyle='--',
                color='orange')
        ax.set_title(f'Индекс ВМО: {idx}')
        ax.grid(True, linestyle='--', alpha=0.5)
        if i % n_cols == 0:
            ax.set_ylabel('Температура (°C)')
        ax.legend(fontsize=8)

    # Скрываем лишние пустые графики
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Предсказательные модели

### XGBRegressor


In [84]:
# Создаем DMatrix для XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Параметры модели
params = {
    'objective': 'reg:squarederror',  # регрессия
    'eval_metric': 'rmse',
    'eta': 0.1,  # learning rate
    'max_depth': 6,  # глубина деревьев
    'subsample': 0.8,  # случайная подвыборка для предотвращения переобучения
    'colsample_bytree': 0.8,
    'seed': 42
}

# Обучение с ранней остановкой
evals = [(dtrain, 'train'), (dval, 'validation')]
xgb_model = xgb.train(
    params,
    dtrain,
    num_boost_round=500,
    evals=evals,
    early_stopping_rounds=20,
    verbose_eval=50
)

# Предсказание на валидации
y_pred = xgb_model.predict(dval)
xgb_result = simple_metric(y_val, y_pred)
xgb_result

[0]	train-rmse:11.54634	validation-rmse:11.34360
[50]	train-rmse:4.08415	validation-rmse:3.07347
[100]	train-rmse:3.80566	validation-rmse:3.06425
Простые метрики:
MAE: 2.6007
MSE: 9.3896
RMSE: 3.0643
MAPE: 14.61%
R²: 0.2955


{'MAE': 2.6006668227059504,
 'MSE': 9.38964255579678,
 'RMSE': np.float64(3.0642523649002507),
 'MAPE': np.float64(14.61413146436517),
 'R2': 0.295545255461527}

### LightGBM

In [85]:
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val)

# Параметры (максимально близко к XGBoost)
params = {
    'objective': 'regression',  # аналог reg:squarederror
    'metric': 'rmse',
    'learning_rate': 0.1,  # eta
    'max_depth': 6,
    'feature_fraction': 0.8,  # colsample_bytree
    'bagging_fraction': 0.8,  # subsample
    'bagging_freq': 1,
    'seed': 42,
    'verbosity': -1
}

# Обучение с early stopping
model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'validation'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20),
        lgb.log_evaluation(period=50)
    ]
)

# Предсказание
y_pred = model.predict(X_val, num_iteration=model.best_iteration)

# Метрика
lgb_result = simple_metric(y_val, y_pred)
lgb_result

Training until validation scores don't improve for 20 rounds
[50]	train's rmse: 4.257	validation's rmse: 3.02647
Early stopping, best iteration is:
[70]	train's rmse: 4.15235	validation's rmse: 3.00598
Простые метрики:
MAE: 2.5415
MSE: 9.0359
RMSE: 3.0060
MAPE: 14.39%
R²: 0.3221


{'MAE': 2.541459484095149,
 'MSE': 9.035923921949781,
 'RMSE': np.float64(3.0059813575519363),
 'MAPE': np.float64(14.39138286525754),
 'R2': 0.32208287586235385}

### HistGradientBoostingRegressor

In [86]:
model = HistGradientBoostingRegressor(
    loss='squared_error',
    learning_rate=0.1,
    max_depth=6,
    max_iter=500,
    max_bins=255,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.2,  # ← ВАЖНО
    random_state=42,
    verbose=1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)

hist_result = simple_metric(y_val, y_pred)
hist_result

Binning 0.039 GB of training data: 0.225 s
Binning 0.010 GB of validation data: 0.060 s
Fitting gradient boosted rounds:
Fit 500 trees in 57.409 s, (15475 total leaves)
Time spent computing histograms: 21.543s
Time spent finding best splits:  11.893s
Time spent applying splits:      19.845s
Time spent predicting:           0.478s
Простые метрики:
MAE: 2.4077
MSE: 8.5507
RMSE: 2.9242
MAPE: 13.73%
R²: 0.3585


{'MAE': 2.407699660851499,
 'MSE': 8.55066199457306,
 'RMSE': np.float64(2.9241514999351623),
 'MAPE': np.float64(13.726620612455346),
 'R2': 0.3584894871953245}

### CatBoostRegressor

In [87]:
# Модель (параметры максимально близки к XGBoost)
model = CatBoostRegressor(
    loss_function='RMSE',
    learning_rate=0.1,
    depth=6,
    iterations=500,
    subsample=0.8,
    rsm=0.8,
    random_seed=42,
    verbose=50
)

# Обучение с early stopping
model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=20,
    use_best_model=True
)

# Предсказание
y_pred = model.predict(X_val)

# Метрика
cat_result = simple_metric(y_val, y_pred)
cat_result

0:	learn: 11.5926656	test: 11.5071894	best: 11.5071894 (0)	total: 33.8ms	remaining: 16.9s
50:	learn: 4.5142100	test: 2.9473925	best: 2.9473925 (50)	total: 1.6s	remaining: 14.1s
100:	learn: 4.3743770	test: 2.8628060	best: 2.8613023 (94)	total: 3s	remaining: 11.8s
150:	learn: 4.2463341	test: 2.8213111	best: 2.8135596 (145)	total: 4.42s	remaining: 10.2s
Stopped by overfitting detector  (20 iterations wait)

bestTest = 2.813559608
bestIteration = 145

Shrink model to first 146 iterations.
Простые метрики:
MAE: 2.3743
MSE: 7.9161
RMSE: 2.8136
MAPE: 13.49%
R²: 0.4061


{'MAE': 2.374281653490453,
 'MSE': 7.916117652026665,
 'RMSE': np.float64(2.8135596052023963),
 'MAPE': np.float64(13.49323115256662),
 'R2': 0.40609596103823875}

## Результаты

In [88]:
results = []

if 'xgb_result' in globals():
    results.append({'model': 'XGBoost', **xgb_result})

if 'lgb_result' in globals():
    results.append({'model': 'LightGBM', **lgb_result})

if 'hist_result' in globals():
    results.append({'model': 'HistGB', **hist_result})

if 'cat_result' in globals():
    results.append({'model': 'CatBoost', **cat_result})


df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='MAE').reset_index(drop=True)

df_results

,model,MAE,MSE,RMSE,MAPE,R2
0,CatBoost,2.374282,7.916118,2.813560,13.493231,0.406096
1,HistGB,2.407700,8.550662,2.924151,13.726621,0.358489
2,LightGBM,2.541459,9.035924,3.005981,14.391383,0.322083
3,XGBoost,2.600667,9.389643,3.064252,14.614131,0.295545


## Подбор параметров

In [89]:
param_grid = {
    'learning_rate': [0.03, 0.05, 0.1],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [3, 5],
    'subsample': [0.7, 0.8],
    'rsm': [0.7, 0.8],
    'random_strength': [1, 2]
}

keys, values = zip(*param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

best_score = float('inf')
best_params = None
best_model = None

for i, params_update in enumerate(param_combinations):
    print(f"\n🔹 Iteration {i+1}/{len(param_combinations)}")

    model = CatBoostRegressor(
        loss_function='RMSE',
        iterations=500,
        random_seed=42,
        verbose=False,
        **params_update
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=20,
        use_best_model=True,
        verbose=False
    )

    y_pred = model.predict(X_val)
    score = simple_metric(y_val, y_pred)['MAE']

    print(f"Score: {score}")
    print(f"Params: {params_update}")

    if score < best_score:
        best_score = score
        best_params = params_update
        best_model = model

print("\nBEST RESULT")
print("Score:", best_score)
print("Params:", best_params)


🔹 Iteration 1/144
Простые метрики:
MAE: 2.3917
MSE: 8.3790
RMSE: 2.8947
MAPE: 13.55%
R²: 0.3714
Score: 2.391701716792767
Params: {'learning_rate': 0.03, 'depth': 4, 'l2_leaf_reg': 3, 'subsample': 0.7, 'rsm': 0.7, 'random_strength': 1}

🔹 Iteration 2/144
Простые метрики:
MAE: 2.4698
MSE: 8.8581
RMSE: 2.9763
MAPE: 13.92%
R²: 0.3354
Score: 2.4698362948910777
Params: {'learning_rate': 0.03, 'depth': 4, 'l2_leaf_reg': 3, 'subsample': 0.7, 'rsm': 0.7, 'random_strength': 2}

🔹 Iteration 3/144
Простые метрики:
MAE: 2.3591
MSE: 7.9778
RMSE: 2.8245
MAPE: 13.28%
R²: 0.4015
Score: 2.359062411888947
Params: {'learning_rate': 0.03, 'depth': 4, 'l2_leaf_reg': 3, 'subsample': 0.7, 'rsm': 0.8, 'random_strength': 1}

🔹 Iteration 4/144
Простые метрики:
MAE: 2.4001
MSE: 8.3547
RMSE: 2.8904
MAPE: 13.60%
R²: 0.3732
Score: 2.4000588589795413
Params: {'learning_rate': 0.03, 'depth': 4, 'l2_leaf_reg': 3, 'subsample': 0.7, 'rsm': 0.8, 'random_strength': 2}

🔹 Iteration 5/144
Простые метрики:
MAE: 2.3950
MSE: 8

In [ ]:
# BEST RESULT
# Score: 2.3242527801397133
# Params: {'learning_rate': 0.1, 'depth': 4, 'l2_leaf_reg': 5, 'subsample': 0.7, 'rsm': 0.8, 'random_strength': 2}